# **Genetic Algorithm: Portfolio Selection & Optimization**
---

This notebook implements a **Genetic Algorithm (GA)** to solve the portfolio selection and
weight optimization problem. Unlike K-Means clustering (which groups similar assets) or
the Efficient Frontier (which optimizes weights for a fixed set of assets), the GA
simultaneously **selects which stocks to include** and **optimizes their weights**.

**Approach:**
- The GA evolves a population of candidate portfolios over multiple generations.
- Each individual encodes both a **subset of stocks** and their **allocation weights**.
- The fitness function evaluates portfolios based on the insurer's mandate:
  low volatility, controlled beta, and reasonable returns.

**Train/Test Split:**
- **Training (2022–2023):** The GA optimizes portfolios using only this data.
- **Testing (2024):** Out-of-sample backtest to validate the strategy.

## **1. Imports & Data Loading**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import random
import warnings
from copy import deepcopy

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
np.random.seed(42)
random.seed(42)

### **1.1 Load Datasets**

In [ ]:
# Load stock price data
stock_data = pd.read_csv('../Data/stock_data.csv', index_col=0, parse_dates=True)

# Load raw dataset (9 features per stock)
raw_dataset = pd.read_csv('../Data/raw_dataset.csv', index_col=0)

# Load fundamentals
fundamentals = pd.read_csv('../Data/fundamentals.csv', index_col=0)

print(f"Stock data shape: {stock_data.shape}")
print(f"Raw dataset shape: {raw_dataset.shape}")
print(f"Fundamentals shape: {fundamentals.shape}")
print(f"\nStock data period: {stock_data.index[0].date()} to {stock_data.index[-1].date()}")

### **1.2 Compute Daily Returns & Train/Test Split**

We split the data into:
- **Training period (2022-01-01 to 2023-12-31):** Used by the GA to evolve portfolios.
- **Test period (2024-01-01 to 2024-12-31):** Out-of-sample validation.

In [ ]:
# Compute daily log returns
log_returns = np.log(stock_data / stock_data.shift(1)).dropna()

# Train/Test split
cutoff_date = '2024-01-01'

train_returns = log_returns.loc[log_returns.index < cutoff_date]
test_returns = log_returns.loc[log_returns.index >= cutoff_date]

print(f"Training period: {train_returns.index[0].date()} to {train_returns.index[-1].date()} ({len(train_returns)} days)")
print(f"Testing period:  {test_returns.index[0].date()} to {test_returns.index[-1].date()} ({len(test_returns)} days)")

### **1.3 Define Valid Tickers**

We restrict the universe to tickers that:
1. Have complete price data (no NaN in training period).
2. Appear in the `raw_dataset.csv` (i.e., have all 9 features computed).

In [ ]:
# Tickers with complete training data
valid_price_tickers = train_returns.columns[train_returns.isna().sum() == 0].tolist()

# Tickers with complete features
valid_feature_tickers = raw_dataset.dropna().index.tolist()

# Intersection
valid_tickers = sorted(list(set(valid_price_tickers) & set(valid_feature_tickers)))

print(f"Valid tickers for GA: {len(valid_tickers)}")

### **1.4 Precompute Training Metrics**

For the fitness evaluation, we precompute annualized metrics on the training period.

In [ ]:
# Precompute training statistics per ticker
train_ann_returns = train_returns[valid_tickers].mean() * 252
train_ann_vol = train_returns[valid_tickers].std() * np.sqrt(252)

# Beta from fundamentals (or raw_dataset)
stock_betas = raw_dataset.loc[valid_tickers, 'beta'].to_dict()

# Covariance matrix (annualized) on training data
train_cov = train_returns[valid_tickers].cov() * 252

print("Training metrics computed.")
print(f"  Mean annualized return: {train_ann_returns.mean():.4f}")
print(f"  Mean annualized vol:    {train_ann_vol.mean():.4f}")

## **2. Genetic Algorithm Design**
---

### **2.1 GA Parameters**

The parameters are calibrated for an insurance mandate:
- **Portfolio size:** 20–30 stocks for diversification.
- **Target beta:** ~0.6 (defensive).
- **Fitness:** Penalizes high volatility heavily (insurer's mandate).

In [ ]:
# ==========================================
# GA PARAMETERS
# ==========================================

PORTFOLIO_SIZE = 25          # Number of stocks in each portfolio
POPULATION_SIZE = 150        # Population of candidate portfolios
GENERATIONS = 80             # Number of evolutionary iterations
ELITE_RATIO = 0.15           # Top 15% are kept as elites
MUTATION_RATE = 0.4          # Probability of mutation
MUTATION_INTENSITY = 3       # Max number of stocks swapped per mutation
WEIGHT_MUTATION_STD = 0.05   # Std dev for weight perturbation
CROSSOVER_RATE = 0.8         # Probability of crossover
TOURNAMENT_SIZE = 5          # Tournament selection size
RISK_FREE_RATE = 0.04        # 4% (US Treasury 1Y approx.)

# Fitness weights (Insurance mandate: LOW RISK)
W_SHARPE = 0.35              # Reward Sharpe Ratio
W_VOLATILITY = 0.35          # Penalize high portfolio volatility
W_BETA = 0.15                # Penalize deviation from target beta
W_DIVERSIFICATION = 0.15     # Reward diversification (low max weight)

TARGET_BETA = 0.6            # Defensive beta target
MAX_SINGLE_WEIGHT = 0.10     # No single stock > 10% (concentration limit)
MAX_VOLATILITY = 0.20        # Volatility ceiling for insurer

print("GA Parameters configured.")
print(f"  Portfolio size: {PORTFOLIO_SIZE} stocks")
print(f"  Population:     {POPULATION_SIZE}")
print(f"  Generations:    {GENERATIONS}")
print(f"  Target Beta:    {TARGET_BETA}")

### **2.2 Individual Representation**

Each individual (chromosome) is a tuple:
- `tickers`: list of selected stock tickers (length = PORTFOLIO_SIZE)
- `weights`: numpy array of portfolio weights (sum = 1, all >= 0)

In [ ]:
def create_individual():
    """Create a random portfolio individual."""
    tickers = random.sample(valid_tickers, PORTFOLIO_SIZE)
    # Random weights, normalized to sum to 1
    weights = np.random.dirichlet(np.ones(PORTFOLIO_SIZE))
    return {'tickers': tickers, 'weights': weights}


def normalize_weights(weights):
    """Ensure weights sum to 1 and are non-negative."""
    weights = np.maximum(weights, 0.0)
    total = weights.sum()
    if total == 0:
        return np.ones(len(weights)) / len(weights)
    return weights / total

### **2.3 Fitness Function**

The fitness function evaluates each portfolio on the **training data** only.
It combines multiple objectives aligned with the insurer's mandate:

$$\text{Fitness} = w_s \cdot \text{Sharpe} - w_v \cdot \text{VolPenalty} - w_\beta \cdot \text{BetaPenalty} + w_d \cdot \text{DivBonus}$$

In [ ]:
def evaluate_fitness(individual):
    """
    Evaluate the fitness of a portfolio individual on training data.
    
    Returns: dict with fitness score and component metrics.
    """
    tickers = individual['tickers']
    weights = individual['weights']
    
    # --- Portfolio Return & Volatility ---
    # Weighted daily returns
    port_daily_returns = (train_returns[tickers] * weights).sum(axis=1)
    
    ann_return = port_daily_returns.mean() * 252
    ann_vol = port_daily_returns.std() * np.sqrt(252)
    
    # Sharpe Ratio
    sharpe = (ann_return - RISK_FREE_RATE) / ann_vol if ann_vol > 0 else -10
    
    # --- Portfolio Beta ---
    port_beta = sum(weights[i] * stock_betas.get(tickers[i], 1.0)
                    for i in range(len(tickers)))
    beta_penalty = abs(port_beta - TARGET_BETA)
    
    # --- Volatility Penalty ---
    # Extra penalty if volatility exceeds the insurer's ceiling
    vol_penalty = max(0, ann_vol - MAX_VOLATILITY) * 5 + ann_vol
    
    # --- Diversification Bonus ---
    # Reward even distribution (penalize concentration)
    max_weight = weights.max()
    concentration_penalty = max(0, max_weight - MAX_SINGLE_WEIGHT) * 10
    hhi = np.sum(weights ** 2)  # Herfindahl index (lower = more diversified)
    div_bonus = 1 / (1 + hhi * PORTFOLIO_SIZE)  # Normalized [0, 1]
    
    # --- Max Drawdown (training) ---
    cum_returns = (1 + port_daily_returns).cumprod()
    peak = cum_returns.cummax()
    drawdown = ((cum_returns - peak) / peak).min()
    
    # --- Combined Fitness ---
    fitness = (
        W_SHARPE * sharpe
        - W_VOLATILITY * vol_penalty
        - W_BETA * beta_penalty
        + W_DIVERSIFICATION * div_bonus
        - concentration_penalty
    )
    
    # Heavy penalty for negative returns
    if ann_return < 0:
        fitness -= 2.0
    
    return {
        'fitness': fitness,
        'ann_return': ann_return,
        'ann_vol': ann_vol,
        'sharpe': sharpe,
        'beta': port_beta,
        'max_drawdown': drawdown,
        'max_weight': max_weight,
        'hhi': hhi
    }

### **2.4 Genetic Operators**

In [ ]:
def tournament_selection(scored_population, k=TOURNAMENT_SIZE):
    """Select an individual via tournament selection."""
    tournament = random.sample(scored_population, k)
    tournament.sort(key=lambda x: x['fitness'], reverse=True)
    return deepcopy(tournament[0]['individual'])


def crossover(parent1, parent2):
    """
    Two-point crossover on both tickers and weights.
    Produces one child.
    """
    if random.random() > CROSSOVER_RATE:
        return deepcopy(parent1)
    
    # --- Ticker crossover ---
    cut1 = random.randint(3, PORTFOLIO_SIZE // 2)
    cut2 = random.randint(PORTFOLIO_SIZE // 2 + 1, PORTFOLIO_SIZE - 3)
    
    child_tickers = parent1['tickers'][:cut1]
    child_weights = list(parent1['weights'][:cut1])
    
    # Add from parent2 (avoid duplicates)
    for i, t in enumerate(parent2['tickers']):
        if t not in child_tickers and len(child_tickers) < PORTFOLIO_SIZE:
            child_tickers.append(t)
            child_weights.append(parent2['weights'][i])
    
    # Fill remaining from parent1 middle section
    for i in range(cut1, cut2):
        t = parent1['tickers'][i]
        if t not in child_tickers and len(child_tickers) < PORTFOLIO_SIZE:
            child_tickers.append(t)
            child_weights.append(parent1['weights'][i])
    
    # If still not enough, add random valid tickers
    while len(child_tickers) < PORTFOLIO_SIZE:
        new_t = random.choice(valid_tickers)
        if new_t not in child_tickers:
            child_tickers.append(new_t)
            child_weights.append(1.0 / PORTFOLIO_SIZE)
    
    child_weights = normalize_weights(np.array(child_weights[:PORTFOLIO_SIZE]))
    
    return {'tickers': child_tickers[:PORTFOLIO_SIZE], 'weights': child_weights}


def mutate(individual):
    """
    Mutate an individual:
    1. Swap some stocks (ticker mutation)
    2. Perturb weights (weight mutation)
    """
    ind = deepcopy(individual)
    
    # --- Ticker mutation ---
    if random.random() < MUTATION_RATE:
        nb_swaps = random.randint(1, MUTATION_INTENSITY)
        for _ in range(nb_swaps):
            idx_remove = random.randint(0, PORTFOLIO_SIZE - 1)
            new_ticker = random.choice(valid_tickers)
            attempts = 0
            while new_ticker in ind['tickers'] and attempts < 50:
                new_ticker = random.choice(valid_tickers)
                attempts += 1
            if new_ticker not in ind['tickers']:
                ind['tickers'][idx_remove] = new_ticker
    
    # --- Weight mutation ---
    if random.random() < MUTATION_RATE:
        noise = np.random.normal(0, WEIGHT_MUTATION_STD, PORTFOLIO_SIZE)
        ind['weights'] = normalize_weights(ind['weights'] + noise)
    
    return ind

### **2.5 Evolution Loop**

In [ ]:
print("=" * 60)
print("GENETIC ALGORITHM - PORTFOLIO OPTIMIZATION")
print("=" * 60)
print(f"Universe: {len(valid_tickers)} stocks")
print(f"Portfolio size: {PORTFOLIO_SIZE}")
print(f"Population: {POPULATION_SIZE} | Generations: {GENERATIONS}")
print(f"Mandate: Target β={TARGET_BETA}, Max Vol={MAX_VOLATILITY:.0%}")
print("=" * 60)

# Initialize population
population = [create_individual() for _ in range(POPULATION_SIZE)]

# Track evolution
history = {
    'best_fitness': [],
    'avg_fitness': [],
    'best_sharpe': [],
    'best_vol': [],
    'best_return': [],
    'best_beta': []
}

best_ever = None

for gen in range(GENERATIONS):
    # Evaluate all individuals
    scored_pop = []
    for ind in population:
        metrics = evaluate_fitness(ind)
        scored_pop.append({
            'individual': ind,
            **metrics
        })
    
    # Sort by fitness (descending)
    scored_pop.sort(key=lambda x: x['fitness'], reverse=True)
    
    # Track best of generation
    best_gen = scored_pop[0]
    avg_fitness = np.mean([x['fitness'] for x in scored_pop])
    
    history['best_fitness'].append(best_gen['fitness'])
    history['avg_fitness'].append(avg_fitness)
    history['best_sharpe'].append(best_gen['sharpe'])
    history['best_vol'].append(best_gen['ann_vol'])
    history['best_return'].append(best_gen['ann_return'])
    history['best_beta'].append(best_gen['beta'])
    
    # Update best ever
    if best_ever is None or best_gen['fitness'] > best_ever['fitness']:
        best_ever = deepcopy(best_gen)
    
    # Print progress every 10 generations
    if gen % 10 == 0 or gen == GENERATIONS - 1:
        print(f"Gen {gen+1:3d}/{GENERATIONS} | "
              f"Fitness: {best_gen['fitness']:+.4f} | "
              f"Sharpe: {best_gen['sharpe']:.3f} | "
              f"Return: {best_gen['ann_return']*100:+.2f}% | "
              f"Vol: {best_gen['ann_vol']*100:.2f}% | "
              f"β: {best_gen['beta']:.3f} | "
              f"Avg: {avg_fitness:+.4f}")
    
    # --- Create next generation ---
    n_elites = max(2, int(POPULATION_SIZE * ELITE_RATIO))
    elites = [deepcopy(x['individual']) for x in scored_pop[:n_elites]]
    
    new_population = elites[:]
    
    while len(new_population) < POPULATION_SIZE:
        parent1 = tournament_selection(scored_pop)
        parent2 = tournament_selection(scored_pop)
        child = crossover(parent1, parent2)
        child = mutate(child)
        new_population.append(child)
    
    population = new_population

print("\n" + "=" * 60)
print("EVOLUTION COMPLETE")
print("=" * 60)

### **2.6 Best Portfolio Summary**

In [ ]:
# Extract best portfolio
best_tickers = best_ever['individual']['tickers']
best_weights = best_ever['individual']['weights']

# Create a clean summary DataFrame
portfolio_df = pd.DataFrame({
    'Ticker': best_tickers,
    'Weight': best_weights,
    'Weight (%)': best_weights * 100,
    'Ann. Return (Train)': [train_ann_returns.get(t, 0) for t in best_tickers],
    'Ann. Vol (Train)': [train_ann_vol.get(t, 0) for t in best_tickers],
    'Beta': [stock_betas.get(t, 1.0) for t in best_tickers]
}).sort_values('Weight (%)', ascending=False).reset_index(drop=True)

print("=" * 60)
print("OPTIMAL PORTFOLIO (GA)")
print("=" * 60)
print(f"\nFitness Score:      {best_ever['fitness']:+.4f}")
print(f"Sharpe Ratio:       {best_ever['sharpe']:.4f}")
print(f"Ann. Return (Train):{best_ever['ann_return']*100:+.2f}%")
print(f"Ann. Volatility:    {best_ever['ann_vol']*100:.2f}%")
print(f"Portfolio Beta:     {best_ever['beta']:.3f}")
print(f"Max Drawdown:       {best_ever['max_drawdown']*100:.2f}%")
print(f"HHI (concentration):{best_ever['hhi']:.4f}")
print(f"\nTop 10 Holdings:")
display(portfolio_df.head(10))

## **3. Evolution Visualization**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Fitness evolution
ax = axes[0, 0]
ax.plot(history['best_fitness'], label='Best Fitness', linewidth=2, color='#2ca02c')
ax.plot(history['avg_fitness'], label='Average Fitness', linewidth=1.5, alpha=0.7, color='#7f7f7f')
ax.set_title('Fitness Evolution', fontsize=14, fontweight='bold')
ax.set_xlabel('Generation')
ax.set_ylabel('Fitness Score')
ax.legend()
ax.grid(True, alpha=0.3)

# Sharpe evolution
ax = axes[0, 1]
ax.plot(history['best_sharpe'], linewidth=2, color='#1f77b4')
ax.set_title('Best Sharpe Ratio per Generation', fontsize=14, fontweight='bold')
ax.set_xlabel('Generation')
ax.set_ylabel('Sharpe Ratio')
ax.grid(True, alpha=0.3)

# Return & Vol evolution
ax = axes[1, 0]
ax.plot([r * 100 for r in history['best_return']], label='Return (%)', linewidth=2, color='#2ca02c')
ax.plot([v * 100 for v in history['best_vol']], label='Volatility (%)', linewidth=2, color='#d62728')
ax.axhline(y=MAX_VOLATILITY * 100, color='#d62728', linestyle='--', alpha=0.5, label=f'Vol Ceiling ({MAX_VOLATILITY:.0%})')
ax.set_title('Return & Volatility Evolution', fontsize=14, fontweight='bold')
ax.set_xlabel('Generation')
ax.set_ylabel('%')
ax.legend()
ax.grid(True, alpha=0.3)

# Beta evolution
ax = axes[1, 1]
ax.plot(history['best_beta'], linewidth=2, color='#ff7f0e')
ax.axhline(y=TARGET_BETA, color='black', linestyle='--', alpha=0.5, label=f'Target β = {TARGET_BETA}')
ax.set_title('Portfolio Beta Evolution', fontsize=14, fontweight='bold')
ax.set_xlabel('Generation')
ax.set_ylabel('Beta')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Genetic Algorithm - Evolution Tracking', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### **3.1 Portfolio Composition**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Weight distribution (Top 15)
ax = axes[0]
top_n = 15
top_df = portfolio_df.head(top_n)
colors = plt.cm.viridis(np.linspace(0.2, 0.8, top_n))
ax.barh(range(top_n), top_df['Weight (%)'], color=colors)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_df['Ticker'])
ax.invert_yaxis()
ax.set_xlabel('Weight (%)')
ax.set_title(f'Top {top_n} Holdings', fontsize=14, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)

# Sector-like: Beta vs Weight scatter
ax = axes[1]
scatter = ax.scatter(portfolio_df['Beta'], portfolio_df['Weight (%)'],
                     s=portfolio_df['Weight (%)'] * 30,
                     c=portfolio_df['Ann. Return (Train)'],
                     cmap='RdYlGn', edgecolors='black', linewidth=0.5, alpha=0.8)
for _, row in portfolio_df.iterrows():
    if row['Weight (%)'] > 3:
        ax.annotate(row['Ticker'], (row['Beta'], row['Weight (%)']),
                    fontsize=8, ha='center', va='bottom')
ax.axvline(x=TARGET_BETA, color='red', linestyle='--', alpha=0.5, label=f'Target β={TARGET_BETA}')
ax.set_xlabel('Beta')
ax.set_ylabel('Weight (%)')
ax.set_title('Holdings: Beta vs Weight (color = Train Return)', fontsize=14, fontweight='bold')
ax.legend()
plt.colorbar(scatter, ax=ax, label='Ann. Return (Train)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## **4. Out-of-Sample Backtest (2024)**
---

This is the critical validation step. The GA has **never seen** 2024 data.
We compare the GA portfolio against the equally-weighted S&P 500 benchmark.

In [ ]:
print("=" * 60)
print("OUT-OF-SAMPLE BACKTEST (2024)")
print("=" * 60)

# --- GA Portfolio on test data ---
# Check which tickers are available in test data
available_test = [t for t in best_tickers if t in test_returns.columns]
missing_test = [t for t in best_tickers if t not in test_returns.columns]

if missing_test:
    print(f"Warning: {len(missing_test)} tickers missing from test data: {missing_test}")

# Re-normalize weights for available tickers
mask = np.array([t in available_test for t in best_tickers])
test_weights = best_weights[mask]
test_weights = test_weights / test_weights.sum()

# GA portfolio daily returns (weighted)
ga_daily_returns = (test_returns[available_test] * test_weights).sum(axis=1)

# --- Benchmark: Equally-weighted universe ---
# Use all valid tickers equally weighted as benchmark
benchmark_tickers = [t for t in valid_tickers if t in test_returns.columns]
benchmark_daily_returns = test_returns[benchmark_tickers].mean(axis=1)

# --- Cumulative performance (Base 100) ---
ga_cumulative = (1 + ga_daily_returns).cumprod() * 100
benchmark_cumulative = (1 + benchmark_daily_returns).cumprod() * 100

# --- Drawdown ---
def calculate_drawdown(cum_series):
    peak = cum_series.cummax()
    return (cum_series - peak) / peak

ga_drawdown = calculate_drawdown(ga_cumulative)
benchmark_drawdown = calculate_drawdown(benchmark_cumulative)

### **4.1 Performance Chart**

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

# --- Performance ---
ax1.plot(ga_cumulative.index, ga_cumulative,
         label='GA Portfolio (Out-of-Sample)', color='#2ca02c', linewidth=2.5)
ax1.plot(benchmark_cumulative.index, benchmark_cumulative,
         label='Equal-Weight Universe (Benchmark)', color='#7f7f7f', linestyle='--', linewidth=1.5, alpha=0.8)

ax1.axhline(y=100, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
ax1.set_title('Out-of-Sample Performance: GA Portfolio vs Benchmark (2024)',
              fontsize=16, fontweight='bold')
ax1.set_ylabel('Portfolio Value (Base 100)', fontsize=12)
ax1.legend(fontsize=12, loc='upper left')
ax1.grid(True, alpha=0.3)

# --- Drawdown ---
ax2.fill_between(ga_drawdown.index, ga_drawdown, 0, color='#d62728', alpha=0.3, label='GA Drawdown')
ax2.plot(ga_drawdown.index, ga_drawdown, color='#d62728', linewidth=1)
ax2.fill_between(benchmark_drawdown.index, benchmark_drawdown, 0, color='gray', alpha=0.15, label='Benchmark Drawdown')
ax2.plot(benchmark_drawdown.index, benchmark_drawdown, color='gray', linewidth=0.5, alpha=0.5)
ax2.set_title('Drawdown Analysis', fontsize=14, fontweight='bold')
ax2.set_ylabel('Drawdown')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### **4.2 Performance Metrics**

In [ ]:
def compute_metrics(daily_returns, label, risk_free=RISK_FREE_RATE):
    """Compute full performance metrics from daily log returns."""
    # Convert to simple returns for cumulative calculations
    simple_returns = np.exp(daily_returns) - 1
    
    cum = (1 + simple_returns).cumprod()
    total_return = cum.iloc[-1] / cum.iloc[0] - 1
    
    ann_return = daily_returns.mean() * 252
    ann_vol = daily_returns.std() * np.sqrt(252)
    sharpe = (ann_return - risk_free) / ann_vol if ann_vol > 0 else 0
    
    # Sortino Ratio
    downside = daily_returns[daily_returns < 0]
    downside_vol = downside.std() * np.sqrt(252) if len(downside) > 0 else ann_vol
    sortino = (ann_return - risk_free) / downside_vol if downside_vol > 0 else 0
    
    # Max Drawdown
    peak = cum.cummax()
    dd = (cum - peak) / peak
    max_dd = dd.min()
    
    # Calmar Ratio
    calmar = ann_return / abs(max_dd) if max_dd != 0 else 0
    
    # Win Rate
    win_rate = (daily_returns > 0).mean()
    
    # Skewness & Kurtosis
    from scipy.stats import skew, kurtosis
    skew_val = skew(daily_returns.dropna())
    kurt_val = kurtosis(daily_returns.dropna())
    
    print(f"\n{'='*45}")
    print(f"  {label}")
    print(f"{'='*45}")
    print(f"  Total Return:       {total_return*100:+.2f}%")
    print(f"  Ann. Return:        {ann_return*100:+.2f}%")
    print(f"  Ann. Volatility:    {ann_vol*100:.2f}%")
    print(f"  Sharpe Ratio:       {sharpe:.4f}")
    print(f"  Sortino Ratio:      {sortino:.4f}")
    print(f"  Max Drawdown:       {max_dd*100:.2f}%")
    print(f"  Calmar Ratio:       {calmar:.4f}")
    print(f"  Win Rate:           {win_rate*100:.1f}%")
    print(f"  Skewness:           {skew_val:.4f}")
    print(f"  Kurtosis:           {kurt_val:.4f}")
    print(f"{'='*45}")
    
    return {
        'total_return': total_return,
        'ann_return': ann_return,
        'ann_vol': ann_vol,
        'sharpe': sharpe,
        'sortino': sortino,
        'max_drawdown': max_dd,
        'calmar': calmar,
        'win_rate': win_rate,
        'skewness': skew_val,
        'kurtosis': kurt_val
    }


ga_metrics = compute_metrics(ga_daily_returns, "GA Portfolio (Out-of-Sample 2024)")
bench_metrics = compute_metrics(benchmark_daily_returns, "Equal-Weight Benchmark (2024)")

### **4.3 Comparative Summary Table**

In [ ]:
comparison_df = pd.DataFrame({
    'GA Portfolio': ga_metrics,
    'EW Benchmark': bench_metrics
}).T

# Format nicely
format_map = {
    'total_return': '{:+.2%}',
    'ann_return': '{:+.2%}',
    'ann_vol': '{:.2%}',
    'sharpe': '{:.4f}',
    'sortino': '{:.4f}',
    'max_drawdown': '{:.2%}',
    'calmar': '{:.4f}',
    'win_rate': '{:.1%}',
    'skewness': '{:.4f}',
    'kurtosis': '{:.4f}'
}

styled_comparison = comparison_df.copy()
for col, fmt in format_map.items():
    if col in styled_comparison.columns:
        styled_comparison[col] = styled_comparison[col].apply(lambda x: fmt.format(x))

print("\n" + "=" * 60)
print("COMPARISON: GA PORTFOLIO vs BENCHMARK")
print("=" * 60)
display(styled_comparison.T)

## **5. Risk Analysis**
---

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Returns Distribution ---
ax = axes[0]
ax.hist(ga_daily_returns, bins=50, alpha=0.7, color='#2ca02c', label='GA Portfolio', density=True)
ax.hist(benchmark_daily_returns, bins=50, alpha=0.5, color='gray', label='Benchmark', density=True)
ax.axvline(x=0, color='black', linestyle='--', linewidth=0.5)
ax.set_title('Daily Returns Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Daily Log Return')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Rolling Volatility (21d) ---
ax = axes[1]
rolling_vol_ga = ga_daily_returns.rolling(21).std() * np.sqrt(252) * 100
rolling_vol_bench = benchmark_daily_returns.rolling(21).std() * np.sqrt(252) * 100
ax.plot(rolling_vol_ga.index, rolling_vol_ga, label='GA Portfolio', color='#2ca02c', linewidth=1.5)
ax.plot(rolling_vol_bench.index, rolling_vol_bench, label='Benchmark', color='gray', linewidth=1, alpha=0.7)
ax.axhline(y=MAX_VOLATILITY * 100, color='red', linestyle='--', alpha=0.5, label=f'Vol Ceiling ({MAX_VOLATILITY:.0%})')
ax.set_title('Rolling 21d Volatility (Ann.)', fontsize=14, fontweight='bold')
ax.set_ylabel('Volatility (%)')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Monthly Returns Heatmap ---
ax = axes[2]
monthly_ga = ga_daily_returns.resample('ME').sum() * 100
monthly_bench = benchmark_daily_returns.resample('ME').sum() * 100
monthly_df = pd.DataFrame({
    'GA': monthly_ga.values[:len(monthly_bench)],
    'Bench': monthly_bench.values
}, index=monthly_bench.index)
monthly_df.index = monthly_df.index.strftime('%b %Y')

sns.heatmap(monthly_df.T, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            ax=ax, cbar_kws={'label': 'Monthly Return (%)'}, linewidths=0.5)
ax.set_title('Monthly Returns (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## **6. Portfolio Details Export**
---

In [ ]:
# Export the final portfolio for comparison with other strategies
portfolio_export = portfolio_df[['Ticker', 'Weight', 'Beta']].copy()
portfolio_export['Strategy'] = 'Genetic Algorithm'

print("\n" + "=" * 60)
print("FINAL GA PORTFOLIO - SELECTED STOCKS")
print("=" * 60)
print(f"\n{PORTFOLIO_SIZE} stocks selected from {len(valid_tickers)} available")
print(f"Portfolio Beta: {best_ever['beta']:.3f} (Target: {TARGET_BETA})")
print(f"Train Sharpe:   {best_ever['sharpe']:.4f}")
print(f"\nSelected tickers:")
print(sorted(best_tickers))

# Save for later comparison
portfolio_export.to_csv('../Data/ga_portfolio.csv', index=False)
print("\nPortfolio saved to Data/ga_portfolio.csv")